# Cell-state defence (HSPC)

Same analysis as MND: per timepoint, cluster cells **without labels** (GTra's `cell_graph_clustering`, Leiden) and compare to the predefined `celltype` annotation (6 hematopoietic progenitor types). High **purity** = each label-free cluster is biologically coherent; ARI is conservative when continuous structure is split more finely than the discrete annotation.

Shared utilities live in the MND folder (`cellstate_utils`, `cellstate_figs`).

In [ ]:
import sys, warnings; warnings.filterwarnings('ignore')
from pathlib import Path
sys.path.insert(0, str(Path('../MND').resolve()))
import numpy as np, pandas as pd, scanpy as sc
import matplotlib.pyplot as plt, seaborn as sns
import cellstate_utils as cu, cellstate_figs as cf

FIG = Path('HSPC_cellstate_figs'); FIG.mkdir(exist_ok=True)
ad = sc.read_h5ad('/data3/projects/2025_GTRA/data/2_HSPC/HSPC_proc.h5ad')
TP = 'time'; ANNOT = 'celltype'; TP_ORDER = ['control', '3h', '24h', '72h']
# HSPC is uploaded via to_df() (no counts layer) -> X is raw counts
summary, labels = cu.evaluate_timepoints(ad, TP, [ANNOT], resolution=0.5, use_counts_layer=None)
summary.to_csv(FIG / 'cellstate_agreement.csv', index=False)
print('mean agreement vs celltype:')
summary[['ARI','AMI','V_measure','purity','inv_purity','homogeneity','completeness']].mean().round(3)

## 1. Agreement per timepoint

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4)); cf.fig_agreement_bars(summary, ANNOT, order=TP_ORDER, ax=ax)
fig.tight_layout(); fig.savefig(FIG / 'CSF1_agreement.pdf', dpi=200); plt.show()
summary.set_index('timepoint').loc[TP_ORDER].reset_index().round(3)

## 2. Confusion matrices (celltype → Leiden cluster, row-normalized)

In [ ]:
fig, axes = plt.subplots(1, len(TP_ORDER), figsize=(4 * len(TP_ORDER), 3.5))
cf.fig_confusion_grid(ad, TP, ANNOT, labels, TP_ORDER, ax=axes)
fig.tight_layout(); fig.savefig(FIG / 'CSF2_confusion.pdf', dpi=200); plt.show()

## 3. UMAP — annotation vs unsupervised Leiden

In [ ]:
fig = cf.fig_tsne_compare(ad, TP, ANNOT, labels, TP_ORDER, coords='X_umap')
fig.savefig(FIG / 'CSF3_umap_compare.pdf', dpi=200); plt.show()

## 4. Resolution sensitivity

In [ ]:
res_df = cu.resolution_scan(ad, TP, ANNOT, resolutions=[0.2, 0.3, 0.5, 0.8, 1.0], use_counts_layer=None)
res_df.to_csv(FIG / 'cellstate_resolution.csv', index=False)
fig, ax = plt.subplots(figsize=(6, 4)); cf.fig_resolution(res_df, ax=ax)
fig.tight_layout(); fig.savefig(FIG / 'CSF4_resolution.pdf', dpi=200); plt.show()